# Lab 13 - Inference Service Optimization

**Textbook:** Chip Huyen, *AI Engineering*, Chapter 9 "Inference Optimization", second half: "Inference Optimization" (Model Optimization: model compression; Inference Service Optimization: batching, prompt caching).
**Environment:** cluster notebook (your local Docker JupyterLab on the KU VPN).
**Time:** Part A about 2.5 hours. This is a **Part-A-only week** (side project 3 is due). Part B is an optional stretch for bonus points.

In Lab 12 you measured how fast the cluster is. This week you make Larkspur's help-desk assistant **faster and cheaper without buying hardware**, using the service-level levers from Chapter 9: prompt caching, batching, output length control, routing easy questions to a small model, and (on paper) quantization. Every claim you make in the analysis should rest on a number you measured here.

### Before you start
- Connect to the **KU VPN** and start your course container.
- Read the second half of Chapter 9 (the section "Inference Optimization": "Model Optimization" and "Inference Service Optimization"). Keep it open for section names.
- The cluster is shared and some cells send long (about 6,000-token) prompts. Run each experiment cell **once**; Part A sends about 75 requests.
- Cells use top-level `await`; run them in order.

## Setup

In [ ]:
import asyncio, time, uuid, re, json, datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from openai import AsyncOpenAI
from itec632 import get_client, MODELS, NO_THINK, settings
from itec632.corpus import load_docs, corpus_dir
from itec632.grading import LabChecker

ck = LabChecker("lab13", part="cluster")
ckb = LabChecker("lab13", part="bonus")    # Part B stretch checks (reported as bonus in the summary)
client = get_client()
base_url, api_key = settings()
aclient = AsyncOpenAI(base_url=base_url, api_key=api_key, timeout=300, max_retries=0)

MOE, DENSE, SMALL, EMBED = MODELS["main"], MODELS["dense"], MODELS["small"], MODELS["embed"]
COLORS = {MOE: "#2a78d6", DENSE: "#eb6834", SMALL: "#1baf7a"}
PAUSE_S = 1.5         # between short requests
LONG_PAUSE_S = 3.0    # between ~6K-token requests (keeps you under 300k tokens/minute)

available = [m.id for m in client.models.list().data]
ck.check("connected", all(m in available for m in [MOE, SMALL, EMBED]), 1,
         "Could not see qwen-moe, qwen-small and embed. Check the VPN and your .env.")
print("Started (UTC):", dt.datetime.now(dt.timezone.utc).strftime("%A %Y-%m-%d %H:%M"))

The timing helper is the same idea as Lab 12 (streaming, `time.perf_counter()`, token counts from `stream_options={"include_usage": True}` with an estimate as fallback). It is complete here. It also records `backend` (which Mac served the request, from the gateway's `x-litellm-model-id` header) and `cached_tokens` if the server reports them.

In [ ]:
def compute_metrics(t_start, t_first, t_end, n_out):
    """Turn the timestamps of one streamed request into latency metrics.

    t_start: perf_counter() just before the request was sent
    t_first: perf_counter() when the first token arrived
    t_end:   perf_counter() when the stream finished
    n_out:   number of output tokens (usage.completion_tokens)
    """
    ttft = t_first - t_start
    total = t_end - t_start
    tpot = (t_end - t_first) / (n_out - 1) if n_out and n_out > 1 else None
    out_tok_per_s = n_out / total if total > 0 else None
    return {"ttft_s": ttft, "tpot_s": tpot, "total_s": total, "out_tok_per_s": out_tok_per_s}

In [ ]:
BUDGET = {"sent": 0}   # counts every request this notebook sends (the cluster is shared!)


def _estimate_tokens(text):
    """Rough fallback: about 4 characters per token for English text."""
    return max(1, round(len(text or "") / 4))


async def measure_stream(model, messages, max_tokens=200, include_usage=True, extra_body=None):
    """Send ONE streamed chat request and time it. Returns a dict (one row of results).

    Timing uses time.perf_counter(), a monotonic high-resolution clock made for
    measuring intervals (time.time() can jump if the system clock is adjusted).
    """
    rec = {"model": model, "max_tokens": max_tokens, "ok": False, "error": None,
           "ttft_s": None, "tpot_s": None, "total_s": None, "out_tok_per_s": None,
           "prompt_tokens": None, "completion_tokens": None, "cached_tokens": None,
           "usage_estimated": False, "est_completion_tokens": None, "n_chunks": 0,
           "finish_reason": None, "backend": None, "text": "", "chunk_times": [],
           "sent_utc": dt.datetime.now(dt.timezone.utc).strftime("%H:%M:%S")}
    kwargs = dict(model=model, messages=messages, max_tokens=max_tokens, stream=True,
                  extra_body=NO_THINK if extra_body is None else extra_body)
    if include_usage:
        # Ask the server to append one final chunk with exact token counts.
        kwargs["stream_options"] = {"include_usage": True}
    BUDGET["sent"] += 1
    usage, parts, t_first = None, [], None
    t_start = time.perf_counter()
    try:
        # with_raw_response gives us the HTTP headers too (which backend Mac served us).
        raw = await aclient.chat.completions.with_raw_response.create(**kwargs)
        rec["backend"] = raw.headers.get("x-litellm-model-id") or raw.headers.get("x-litellm-model-api-base")
        async for chunk in raw.parse():
            now = time.perf_counter()
            if getattr(chunk, "usage", None):
                usage = chunk.usage                      # the final usage chunk
            for choice in chunk.choices or []:          # the usage chunk has no choices
                delta = choice.delta
                piece = getattr(delta, "content", None) or ""
                thought = getattr(delta, "reasoning_content", None) or ""
                if piece or thought:
                    if t_first is None:
                        t_first = now                    # first token of any kind
                    rec["chunk_times"].append(round(now - t_start, 4))
                    parts.append(piece)
                if choice.finish_reason:
                    rec["finish_reason"] = choice.finish_reason
        t_end = time.perf_counter()
    except Exception as e:                               # e.g. RateLimitError (429), timeout
        rec["total_s"] = time.perf_counter() - t_start
        rec["error"] = f"{type(e).__name__}: {e}"[:200]
        return rec
    text = "".join(parts)
    rec["text"], rec["n_chunks"] = text, len(rec["chunk_times"])
    rec["est_completion_tokens"] = _estimate_tokens(text)
    if usage is not None and usage.completion_tokens:
        rec["prompt_tokens"] = usage.prompt_tokens
        rec["completion_tokens"] = usage.completion_tokens
        details = getattr(usage, "prompt_tokens_details", None)
        rec["cached_tokens"] = getattr(details, "cached_tokens", None) if details else None
    else:
        # No usage chunk (stream cut off, or a server that ignores stream_options):
        # fall back to an estimate and flag it so you know the number is approximate.
        rec["usage_estimated"] = True
        rec["prompt_tokens"] = sum(_estimate_tokens(m["content"]) for m in messages)
        rec["completion_tokens"] = rec["est_completion_tokens"]
    if t_first is None:
        rec["error"] = "stream ended without any content"
        rec["total_s"] = t_end - t_start
        return rec
    rec.update(compute_metrics(t_start, t_first, t_end, rec["completion_tokens"]))
    rec["ok"] = True
    return rec


def p50_p90(df, by, cols):
    """Median (p50) and 90th percentile (p90) of each column, per group. Failed requests are skipped."""
    rows = []
    ok = df[df["ok"] == True] if len(df) else df  # noqa: E712
    for key, g in ok.groupby(by):
        row = dict(zip(by, key)) if isinstance(by, list) else {by: key}
        row["n"] = len(g)
        for c in cols:
            s = pd.to_numeric(g[c], errors="coerce").dropna()
            row[f"{c}_p50"] = s.median() if len(s) else np.nan
            row[f"{c}_p90"] = s.quantile(0.9) if len(s) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def label_backends(df):
    """Replace long backend ids from the gateway headers with short labels: mac-A, mac-B, ..."""
    names = {}
    for b in df["backend"]:
        if b and b not in names:
            names[b] = f"mac-{chr(65 + len(names))}"
    return df["backend"].map(lambda b: names.get(b, "unknown"))


def nonce():
    """A short random tag. Putting it at the START of a prompt makes the prompt unique,
    so the server cannot reuse a cached prefix from an earlier request."""
    return uuid.uuid4().hex[:8]

## Part A - Guided practice

### A1. Prompt caching: reuse the prefix, skip the prefill

Larkspur's assistant sends the same long instructions and reference documents with every question; only the short question at the end changes. Chapter 9 ("Prompt caching" under "Inference Service Optimization") describes the fix: store the processed prompt (its **KV cache**) and reuse it when a new prompt **starts with** the same tokens. Prefill was the expensive, compute-bound part of TTFT in Lab 12, so skipping it should cut TTFT sharply.

oMLX implements this with block-based prefix sharing: a hot tier in RAM and a colder tier on SSD. Two things decide whether you benefit:

1. **Order.** A token's KV values depend on every token before it. The server can only reuse the cache up to the **first token that differs**. Shared material must come first; the changing part last.
2. **Which machine.** Each Mac has its own cache. `qwen-moe` runs on two Macs and the gateway picks one at random per request, so a "warm" prompt may land on the Mac that has never seen it. `qwen-small` has only one Mac.

The reference pack below is three current Larkspur documents (price sheet, SLA, service catalog), about 6,000 tokens. Each prompt starts with `Session <tag>`. A fresh tag makes the prompt new, so it is **cold** (no classmate can have cached it either); reusing a tag makes it **warm**.

**TODO:** write `prefix_last()`: same content, but the **question comes before** the reference documents.

In [ ]:
by_id = {d["doc_id"]: d for d in load_docs()}
PACK = "\n\n".join(f"## {by_id[i]['title']} ({i})\n{by_id[i]['text']}" for i in ["PRICE-2026", "SLA-EXH-2026", "SVC-CAT-2026"])[:24000]
INSTRUCTIONS = ("You are Larkspur Ridge IT's help-desk assistant. Answer the question using only the "
                "reference documents. Be brief.")
print(f"Reference pack: {len(PACK):,} characters (about {len(PACK) // 4:,} tokens)")


def prefix_first(tag, question):
    """Shared material FIRST (cache-friendly): instructions + documents, then the question."""
    return [{"role": "system", "content": f"Session {tag}. {INSTRUCTIONS}\n\n=== REFERENCE DOCUMENTS ===\n{PACK}"},
            {"role": "user", "content": question}]

In [ ]:
def prefix_last(tag, question):
    """Question FIRST, then the documents (the cache-unfriendly order)."""
    # TODO: return two messages: a system message f"Session {tag}. {INSTRUCTIONS}" (no documents),
    #       and a user message that contains the question FIRST and then the PACK.
    return None

In [ ]:
_t = prefix_last("abc", "QQQ?")
ok_last = bool(_t) and "QQQ?" in _t[-1]["content"] and PACK[:200] in _t[-1]["content"] \
          and _t[-1]["content"].index("QQQ?") < _t[-1]["content"].index(PACK[:200]) and PACK[:200] not in _t[0]["content"]
ck.check("prefix_last_built", ok_last, 2,
         "prefix_last() must return [system, user] with the question before PACK in the user message, and no PACK in system.");

Now the experiment (about 21 long requests, 2-3 minutes). We only ask for 40 output tokens because we care about TTFT.

- **cold**: a new tag every time, documents first.
- **warm**: one "prime" request with tag W, then 6 different questions with the same tag W.
- **last**: question before documents, one prime, then 4 questions with the same tag.
- **qwen-small** warm vs cold, for a model with a single backend.

In [ ]:
CACHE_QS = [
    "What is the support phone number?", "How much is the Premier tier per user?", "What are the payment terms?",
    "What is the trip charge in the metro?", "What is the PagerLine number?", "Which holidays does Larkspur observe?",
    "What is the P2 response target on Essentials?", "How are SLA credits requested?",
    "What is the business-hours labor rate?", "When are invoices sent?",
]
cache_records = []


async def run_cache(model, condition, messages):
    rec = await measure_stream(model, messages, max_tokens=40)
    rec["condition"] = condition
    cache_records.append(rec)
    await asyncio.sleep(LONG_PAUSE_S)
    return rec

for q in CACHE_QS[:4]:                                         # cold
    await run_cache(MOE, "cold", prefix_first(nonce(), q))
WARM_TAG = nonce()                                             # warm (this tag is reused in A3 and A4)
await run_cache(MOE, "prime", prefix_first(WARM_TAG, "Reply with OK."))
for q in CACHE_QS[4:10]:
    await run_cache(MOE, "warm", prefix_first(WARM_TAG, q))
if prefix_last("x", "y") is not None:                         # question first
    LAST_TAG = nonce()
    await run_cache(MOE, "prime_last", prefix_last(LAST_TAG, "Reply with OK."))
    for q in CACHE_QS[:4]:
        await run_cache(MOE, "last", prefix_last(LAST_TAG, q))
else:
    print("Skipping the 'last' condition: complete prefix_last() first.")
await run_cache(SMALL, "cold", prefix_first(nonce(), CACHE_QS[0]))   # single-backend model
await run_cache(SMALL, "prime", prefix_first(WARM_TAG, "Reply with OK."))
for q in CACHE_QS[1:4]:
    await run_cache(SMALL, "warm", prefix_first(WARM_TAG, q))

df_cache = pd.DataFrame(cache_records)
df_cache["backend"] = label_backends(df_cache)
print("errors:", int((~df_cache["ok"]).sum()), "| requests so far:", BUDGET["sent"])
display(df_cache[["model", "condition", "backend", "prompt_tokens", "cached_tokens", "ttft_s", "total_s"]].round(3))

In [ ]:
cache_summary = p50_p90(df_cache, ["model", "condition"], ["ttft_s"])
display(cache_summary.round(3))

fig, ax = plt.subplots(figsize=(8, 3.6))
order = ["cold", "prime", "warm", "prime_last", "last"]
markers = {"mac-A": "o", "mac-B": "s", "unknown": "o"}
for m in [MOE, SMALL]:
    for i, cond in enumerate(order):
        g = df_cache[(df_cache.model == m) & (df_cache.condition == cond) & df_cache.ok]
        for _, row in g.iterrows():
            ax.scatter(i + (0.12 if m == SMALL else -0.12), row.ttft_s, color=COLORS[m],
                       marker=markers.get(row.backend, "o"), s=45, alpha=0.85)
for m in [MOE, SMALL]:
    ax.scatter([], [], color=COLORS[m], label=m)
for b, mk in markers.items():
    if b in set(df_cache.backend) and b != "unknown":
        ax.scatter([], [], color="#777777", marker=mk, label=b)
ax.set_xticks(range(len(order)), order); ax.set_ylabel("TTFT (s)"); ax.set_ylim(bottom=0)
ax.set_title("TTFT by cache condition (marker shape = which Mac served it)"); ax.grid(alpha=0.3, axis="y"); ax.legend(fontsize=8)
plt.show()

**TODO:** compute the speedup (median cold TTFT divided by median warm TTFT) for `qwen-moe`, and look at the warm rows' `backend` column. Did any warm request land on a Mac that had not seen the prefix yet? What was its TTFT?

In [ ]:
def med(model, cond, col="ttft_s"):
    g = df_cache[(df_cache.model == model) & (df_cache.condition == cond) & df_cache.ok]
    return g[col].median() if len(g) else None

cold_moe, warm_moe = med(MOE, "cold"), med(MOE, "warm")
# TODO: cache_speedup = cold_moe / warm_moe  (guard against None)
cache_speedup = None
print("qwen-moe cache speedup (cold / warm median TTFT):", cache_speedup)

In [ ]:
n_cond = df_cache[df_cache.ok].groupby(["model", "condition"]).size()
ck.check("cache_conditions_measured", n_cond.get((MOE, "cold"), 0) >= 3 and n_cond.get((MOE, "warm"), 0) >= 5
         and n_cond.get((MOE, "last"), 0) >= 3, 2,
         "Need >= 3 cold, >= 5 warm and >= 3 'last' qwen-moe requests that succeeded. Complete prefix_last() and re-run.")
ck.check("cache_speedup_computed", cache_speedup is not None, 1, "Compute cache_speedup in the TODO cell.")
ck.check("warm_faster_than_cold", bool(cold_moe and warm_moe and warm_moe < 0.8 * cold_moe), 2,
         "Warm TTFT should be well below cold. Did you reuse the SAME tag for warm requests, with the documents FIRST?")
for (m, cond), v in cache_summary.set_index(["model", "condition"])["ttft_s_p50"].items():
    ck.record(f"ttft_{cond}_{m.replace('qwen-', '')}", float(v))
if cache_speedup:
    ck.record("cache_speedup_moe", float(cache_speedup))

### A2. Batching: sequential vs concurrent

Chapter 9 ("Batching") compares static, dynamic and **continuous batching**. oMLX uses continuous batching: new requests join the running batch as soon as there is a free slot (up to 8 per Mac), and finished ones leave without waiting for the rest. Decode is memory-bandwidth-bound, so one pass over the weights can serve several requests almost as cheaply as one.

For an application this means: if Larkspur needs to process many independent items (summarize 8 tickets, classify a queue), **send them concurrently** instead of in a loop. Here you process the same 8 short requests both ways. Your key allows 4 at a time, so the concurrent version sends two groups of 4.

**TODO:** in the concurrent loop, send each group of 4 at once with `asyncio.gather`.

In [ ]:
TICKETS = [
    "Outlook keeps asking for a password after a password change.", "A shared drive mapping disappeared after a reboot.",
    "The office printer shows offline for everyone.", "A new hire needs a laptop and accounts by Monday.",
    "Teams calls drop every few minutes on Wi-Fi.", "A user deleted a folder on the file server.",
    "The firewall sends a high CPU alert every night.", "A user's OneDrive will not sync.",
]


def ticket_msgs(t):
    return [{"role": "system", "content": f"Request {nonce()}. You are Larkspur Ridge IT's help-desk assistant."},
            {"role": "user", "content": f"Ticket: {t}\nWrite three short troubleshooting steps for a Tier 1 technician."}]

t0 = time.perf_counter()
seq_recs = []
for t in TICKETS:
    seq_recs.append(await measure_stream(MOE, ticket_msgs(t), max_tokens=150))
seq_wall = time.perf_counter() - t0
await asyncio.sleep(5)

t0 = time.perf_counter()
conc_recs = []
for group in (TICKETS[:4], TICKETS[4:]):
    # TODO: results = await asyncio.gather(*[measure_stream(MOE, ticket_msgs(t), max_tokens=150) for t in group])
    results = []
    conc_recs += list(results)
conc_wall = time.perf_counter() - t0

In [ ]:
df_batch = pd.concat([pd.DataFrame(seq_recs).assign(mode="sequential"), pd.DataFrame(conc_recs).assign(mode="concurrent")],
                     ignore_index=True)
walls = {"sequential": seq_wall, "concurrent": conc_wall}
batch_table = p50_p90(df_batch, "mode", ["ttft_s", "total_s", "tpot_s"])
if len(batch_table):
    tok = df_batch[df_batch.ok].groupby("mode")["completion_tokens"].sum()
    batch_table["wall_s"] = batch_table["mode"].map(walls)
    batch_table["throughput_tok_s"] = batch_table["mode"].map(lambda m: tok.get(m, 0) / walls[m])
    display(batch_table.round(3))
ok_counts = df_batch[df_batch.ok].groupby("mode").size() if len(df_batch) else {}
ck.check("batching_ran", ok_counts.get("sequential", 0) == 8 and ok_counts.get("concurrent", 0) == 8, 2,
         "Both modes need 8 successful requests. Complete the asyncio.gather TODO.")
ck.check("concurrent_faster", ok_counts.get("concurrent", 0) == 8 and conc_wall < 0.8 * seq_wall, 2,
         "Concurrent wall time should be clearly below sequential. Are the 4 requests of a group really sent together?")
ck.record("batch_wall_sequential_s", float(seq_wall)); ck.record("batch_wall_concurrent_s", float(conc_wall));

### A3. Output length control: `max_tokens` vs asking for concise answers

Lab 12 showed `total latency = TTFT + TPOT x (output tokens - 1)`. With a warm prompt cache, TTFT is small, so **output length dominates latency**, and output tokens are also the expensive tokens on every price sheet. There are two levers:

- **`max_tokens`** is a hard stop. It bounds latency and cost, but the model does not know about it, so answers are cut off mid-sentence (`finish_reason == "length"`).
- **A concise-answer instruction** changes what the model plans to say. Answers end naturally, but length is not guaranteed.

We use the warm reference pack from A1 (same tag, so TTFT stays low) and these hypothetical prices, in USD per million tokens, so you can talk about cost:

**TODO 1:** write `request_cost()`. **TODO 2:** write your concise-answer instruction in `CONCISE_STYLE`.

In [ ]:
# Hypothetical prices (USD per 1M tokens: input, output), in line with hosted APIs for models of these sizes.
PRICE_PER_M = {MOE: (0.20, 0.80), DENSE: (0.40, 1.60), SMALL: (0.05, 0.20)}


def request_cost(model, prompt_tokens, completion_tokens):
    """Cost in USD of one request at PRICE_PER_M."""
    p_in, p_out = PRICE_PER_M[model]
    # TODO: return (prompt_tokens * p_in + completion_tokens * p_out) / 1_000_000
    return 0.0

# TODO: an instruction that makes answers short (e.g., "Answer in at most two short sentences. ...")
CONCISE_STYLE = ""

In [ ]:
ck.check("request_cost", abs(request_cost(MOE, 1_000_000, 0) - 0.20) < 1e-9 and abs(request_cost(MOE, 0, 1_000_000) - 0.80) < 1e-9, 1,
         "request_cost(model, 1e6, 0) should equal the input price; (model, 0, 1e6) the output price.")
VERBOSE_STYLE = "Answer thoroughly, with background and a full explanation."
CONDITIONS = {
    "verbose":         {"style": VERBOSE_STYLE, "max_tokens": 600},
    "verbose_cap_60":  {"style": VERBOSE_STYLE, "max_tokens": 60},
    "concise":         {"style": CONCISE_STYLE, "max_tokens": 600},
    "concise_cap_120": {"style": CONCISE_STYLE, "max_tokens": 120},
}
LENGTH_QS = ["How does the SLA clock work when Larkspur is waiting on the client?",
             "What does the Professional tier include?",
             "How should a client report a P1 issue, and why?"]

length_records = []
if CONCISE_STYLE.strip():
    for q in LENGTH_QS:
        for name, cfg in CONDITIONS.items():
            rec = await measure_stream(MOE, prefix_first(WARM_TAG, f"{q}\n\n{cfg['style']}"), max_tokens=cfg["max_tokens"])
            rec.update(condition=name, question=q)
            length_records.append(rec)
            await asyncio.sleep(LONG_PAUSE_S)
else:
    print("Write CONCISE_STYLE first.")
df_len = pd.DataFrame(length_records)

In [ ]:
len_table = pd.DataFrame()
if len(df_len) and df_len["ok"].any():
    df_len["cost_usd"] = [request_cost(r.model, r.prompt_tokens, r.completion_tokens) for r in df_len.itertuples()]
    ok = df_len[df_len.ok]
    len_table = ok.groupby("condition").agg(
        out_tokens_p50=("completion_tokens", "median"), total_s_p50=("total_s", "median"), ttft_s_p50=("ttft_s", "median"),
        cost_per_1k_requests_usd=("cost_usd", lambda s: 1000 * s.mean()),
        truncated_share=("finish_reason", lambda s: (s == "length").mean())).reindex(list(CONDITIONS)).reset_index()
    display(len_table.round(4))
    print("How each answer ENDS (look for cut-off sentences):")
    for r in ok[ok.question == LENGTH_QS[1]].itertuples():
        print(f"  {r.condition:16} ...{r.text.strip()[-90:]!r}")
    corr = ok["completion_tokens"].corr(ok["total_s"]) if len(ok) > 2 else None
    print("correlation(output tokens, total latency):", None if corr is None else round(corr, 3))

    cond_colors = dict(zip(CONDITIONS, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]))
    fig, ax = plt.subplots(figsize=(7, 3.6))
    for name in CONDITIONS:
        g = ok[ok.condition == name]
        ax.scatter(g.completion_tokens, g.total_s, color=cond_colors[name], s=45, label=name)
    ax.set_xlabel("output tokens"); ax.set_ylabel("total latency (s)"); ax.set_title("Latency follows output length")
    ax.grid(alpha=0.3); ax.legend(fontsize=8); ax.set_xlim(left=0); ax.set_ylim(bottom=0); plt.show()

In [ ]:
def lv(cond, col):
    g = len_table[len_table.condition == cond] if len(len_table) else []
    return float(g[col].iloc[0]) if len(g) and pd.notna(g[col].iloc[0]) else None

ck.check("length_conditions_ran", len(df_len) and df_len["ok"].sum() >= 12, 2,
         "All 4 conditions x 3 questions should succeed. Write CONCISE_STYLE and re-run.")
v, c = lv("verbose", "out_tokens_p50"), lv("concise", "out_tokens_p50")
ck.check("concise_is_shorter", bool(v and c and c < 0.7 * v), 2,
         "The concise instruction should cut median output tokens well below 'verbose'. Make CONCISE_STYLE stricter.")
for cond in CONDITIONS:
    for col, short in [("out_tokens_p50", "tok"), ("total_s_p50", "lat")]:
        if lv(cond, col) is not None:
            ck.record(f"len_{short}_{cond}", lv(cond, col))

### A4. Model routing as an optimization (light version)

Many help-desk questions are simple lookups ("what is the PagerLine number?") that a 4B model answers as well as the main model, faster and cheaper. Others need several steps of reasoning (an SLA credit calculation). Chapter 10 treats routers as part of the architecture, and Lab 15 builds a real one. Here you just test the economics with a **heuristic**.

Each question in `ROUTE_SET` has a label (`easy`/`hard`) and a few strings a correct answer should contain (a crude quality proxy: it can miss correct answers phrased differently, so read the answers too). The cell runs every question on **both** models (24 requests, warm cache), and then compares three strategies offline: all `qwen-moe`, all `qwen-small`, and routed.

**TODO:** write `is_hard(question)`. Keep it simple: no model calls. Hint: look at what the hard questions have in common.

In [ ]:
ROUTE_SET = [
    {"q": "What is the phone number for the support line?", "label": "easy", "expect": ["555-0111"]},
    {"q": "What is the PagerLine after-hours phone number?", "label": "easy", "expect": ["555-0142"]},
    {"q": "How much does the Professional tier cost per user per month?", "label": "easy", "expect": ["145"]},
    {"q": "What are Larkspur's business hours?", "label": "easy", "expect": ["7:00", "7 am", "7am"]},
    {"q": "What are the payment terms on invoices?", "label": "easy", "expect": ["net 15"]},
    {"q": "What is the trip charge for a billable visit in the Kansas City metro?", "label": "easy", "expect": ["75"]},
    {"q": "A Professional client with 40 users had three missed P1 response targets in May. How much SLA credit do they get, in dollars?",
     "label": "hard", "expect": ["870"]},
    {"q": "An Essentials client submits a P3 ticket at 4:30 PM on a regular Friday (not a holiday). By what day and time is the first response due?",
     "label": "hard", "expect": ["9:30"]},
    {"q": "An Essentials client with 25 users adds VaultLine backup. What is their total monthly cost for managed services plus backup?",
     "label": "hard", "expect": ["2525"]},
    {"q": "An Essentials client needs 30 minutes of out-of-scope remote work on a Saturday. What will it cost?",
     "label": "hard", "expect": ["247.5"]},
    {"q": "A Professional client with 60 users missed five P1 response targets in one month. What is the SLA credit in dollars?",
     "label": "hard", "expect": ["1740"]},
    {"q": "A Premier client opens a P2 ticket at 8:00 PM on a Saturday. When is the response due?",
     "label": "hard", "expect": ["9:00 pm", "9 pm", "9pm", "9:00pm", "21:00"]},
]


def looks_correct(text, expect):
    """Crude proxy: does the answer contain one of the expected strings? (commas and $ removed)"""
    t = (text or "").lower().replace(",", "").replace("$", "")
    return any(e.lower().replace(",", "") in t for e in expect)

In [ ]:
def is_hard(question):
    """True if the question likely needs multi-step reasoning (route to qwen-moe)."""
    # TODO: a simple rule (1-5 lines). Ideas: contains digits? long question? words like "how much ... if"?
    return False

In [ ]:
router_acc = np.mean([is_hard(it["q"]) == (it["label"] == "hard") for it in ROUTE_SET])
print(f"Heuristic agrees with the labels on {router_acc:.0%} of questions")
ck.check("router_heuristic", router_acc >= 10 / 12, 2, "is_hard() should agree with at least 10 of the 12 labels.")

route_records = []
for it in ROUTE_SET:
    for model in [MOE, SMALL]:
        rec = await measure_stream(model, prefix_first(WARM_TAG, it["q"]), max_tokens=300)
        rec.update(q=it["q"], label=it["label"], correct=looks_correct(rec["text"], it["expect"]))
        route_records.append(rec)
        await asyncio.sleep(LONG_PAUSE_S)
df_route = pd.DataFrame(route_records)
print("errors:", int((~df_route["ok"]).sum()), "| requests so far:", BUDGET["sent"])

In [ ]:
tally = pd.DataFrame()
if df_route["ok"].any():
    df_route["cost_usd"] = [request_cost(r.model, r.prompt_tokens or 0, r.completion_tokens or 0) for r in df_route.itertuples()]
    pick = {"all qwen-moe": lambda q: MOE, "all qwen-small": lambda q: SMALL, "routed": lambda q: MOE if is_hard(q) else SMALL}
    rows = []
    for name, choose in pick.items():
        chosen = pd.concat([df_route[(df_route.q == it["q"]) & (df_route.model == choose(it["q"]))] for it in ROUTE_SET])
        rows.append({"strategy": name, "n": len(chosen), "correct": int(chosen.correct.sum()),
                     "mean_latency_s": chosen.total_s.mean(), "cost_per_1k_questions_usd": 1000 * chosen.cost_usd.mean()})
    tally = pd.DataFrame(rows)
    display(tally.round(4))
    print("\nPer-question results (read a few answers yourself; the keyword check is crude):")
    display(df_route.pivot_table(index="q", columns="model", values="correct", aggfunc="first"))

In [ ]:
ck.check("routing_tally", len(tally) == 3 and tally["n"].min() == len(ROUTE_SET), 2,
         "The tally needs all three strategies over all 12 questions. Check df_route['error'].")
if len(tally):
    t = tally.set_index("strategy")
    for name, short in [("all qwen-moe", "moe"), ("all qwen-small", "small"), ("routed", "routed")]:
        ck.record(f"route_correct_{short}", int(t.loc[name, "correct"]))
        ck.record(f"route_cost1k_{short}", float(t.loc[name, "cost_per_1k_questions_usd"]))
        ck.record(f"route_latency_{short}", float(t.loc[name, "mean_latency_s"]))

### A5. Quantization (conceptual, no GPU work)

Chapter 9's "Model compression" section lists quantization as the most common model-level optimization: store weights with fewer bits. In Lab 2 you did the memory math; here you connect it to **speed**. Because decode is memory-bandwidth-bound (Lab 12), halving the bytes per weight roughly halves the bytes read per token, which raises the decode ceiling, while quality can drop, especially at 4 bits and below.

The cluster already made these choices for you: `qwen-moe` is served at about 8 bits, `qwen-dense` at about 4 bits, `qwen-small` at 8 bits. Each Mac lets the GPU use about 72 GB by default.

**TODO:** write `weight_gb()`. The table then shows weight memory and the bandwidth-bound decode ceiling (819 GB/s, active parameters only) at 16, 8 and 4 bits.

In [ ]:
def weight_gb(params_billion, bits):
    """Memory for the weights alone, in GB: parameters x bits / 8 bits per byte."""
    # TODO: return params_billion * bits / 8
    return None

In [ ]:
PEAK_BW_GB_S = 819
SPECS = {MOE: (35, 3), DENSE: (27, 27), SMALL: (4, 4)}      # (total params B, active params B)
quant_rows = []
for m, (total, active) in SPECS.items():
    for bits in (16, 8, 4):
        w, a = weight_gb(total, bits), weight_gb(active, bits)
        quant_rows.append({"model": m, "bits": bits, "weights_gb": w,
                           "fits_72gb": (w is not None and w < 72),
                           "decode_ceiling_tok_s": PEAK_BW_GB_S / a if a else None})
df_quant = pd.DataFrame(quant_rows)
display(df_quant.round(1))
moe_decode = None
if len(df_len) and df_len["ok"].any():
    moe_decode = 1 / df_len[df_len.ok]["tpot_s"].dropna().median()
    print(f"Your measured qwen-moe decode speed this week (A3): {moe_decode:.0f} tokens/s (served at ~8 bits)")
ck.check("quant_memory_math", weight_gb(35, 16) == 70 and weight_gb(27, 4) == 13.5, 2,
         "weight_gb(35, 16) should be 70 GB and weight_gb(27, 4) 13.5 GB.")
if moe_decode:
    ck.record("moe_decode_tok_s", float(moe_decode))

# Optional: paste your Lab 2 numbers here so the analysis can compare them.
LAB2_NOTES = ""

## Part B - Stretch (optional, bonus): a response cache in front of the model

Prompt caching saves prefill. A **response cache** skips the model entirely when the same question comes back. Chapter 10 discusses exact and semantic caches as part of AI application architecture ("Step 4. Reduce Latency with Caches": "Exact caching" and "Semantic caching"); we try one here because it is the biggest latency win of all, if it is right.

- **Exact cache:** normalize the question (lowercase, strip punctuation) and look it up in a dict. Never wrong, rarely hits.
- **Semantic cache:** embed the question with `embed` and reuse a stored answer if the cosine similarity to a past question is at or above a threshold. Hits paraphrases, but can return the **wrong** answer for a question that only looks similar ("Premier P1 response time" vs "Professional P1 response time"). That is a **false hit**.

`helpdesk_replay.jsonl` holds 50 help-desk questions in arrival order: originals, exact repeats, paraphrases, and near-misses. Each has an `intent` label, so we can tell a correct hit from a false one.

Set `RUN_PART_B = True` to run it (about 25-30 requests). **TODO A and B** are in `lookup()`.

In [ ]:
RUN_PART_B = False      # set True to do the optional stretch

In [ ]:
def normalize(q):
    return " ".join(re.sub(r"[^a-z0-9 ]", " ", q.lower()).split())


class ResponseCache:
    def __init__(self, threshold=0.85):
        self.threshold = threshold
        self.exact = {}         # normalized question -> entry
        self.vectors = []       # unit-length embedding of each stored question
        self.entries = []       # entry dicts, same order as vectors

    def add(self, question, vector, answer, intent=None):
        entry = {"question": question, "answer": answer, "intent": intent}
        self.exact[normalize(question)] = entry
        v = np.asarray(vector, dtype=float)
        self.vectors.append(v / np.linalg.norm(v))
        self.entries.append(entry)

    def lookup(self, question, vector):
        """Return (entry, kind, score). kind is 'exact', 'semantic' or 'miss'."""
        # TODO A (exact): if normalize(question) is in self.exact, return (that entry, "exact", 1.0)

        if not self.vectors:
            return None, "miss", 0.0
        # TODO B (semantic): v = unit vector of `vector`; sims = np.vstack(self.vectors) @ v;
        #        best = index of the max; if sims[best] >= self.threshold return (self.entries[best], "semantic", score)
        #        otherwise return (None, "miss", score)
        return None, "miss", 0.0

In [ ]:
# Unit tests on toy vectors (no requests).
_c = ResponseCache(threshold=0.9)
_c.add("What are your hours?", [1, 0, 0], "7-6", "hours")
_c.add("What is the PagerLine number?", [0, 1, 0], "555-0142", "pager")
e1, k1, _ = _c.lookup("what are your HOURS", [0, 0, 1])
e2, k2, s2 = _c.lookup("When are you open?", [0.95, 0.1, 0])
e3, k3, _ = _c.lookup("Something else", [0.5, 0.5, 0.7])
ckb.check("exact_cache", k1 == "exact" and e1 and e1["intent"] == "hours", 2,
          "TODO A: a question that normalizes to a stored one must return kind 'exact'.")
ckb.check("semantic_lookup", k2 == "semantic" and e2 and e2["intent"] == "hours" and k3 == "miss", 3,
          "TODO B: cosine >= threshold should be a 'semantic' hit; a dissimilar vector must be a 'miss'.");

In [ ]:
def find_replay():
    for p in [Path("helpdesk_replay.jsonl"), corpus_dir().parents[1] / "labs" / "lab13-serving-optimization" / "helpdesk_replay.jsonl"]:
        if p.exists():
            return p
    raise FileNotFoundError("helpdesk_replay.jsonl not found; open this notebook from its lab folder.")

REPLAY = [json.loads(line) for line in find_replay().read_text().splitlines() if line.strip()]
print(len(REPLAY), "questions;", pd.Series([r["kind"] for r in REPLAY]).value_counts().to_dict())
THRESHOLD = 0.85
replay_rows, VECS = [], None

if RUN_PART_B:
    # One batch call embeds all 50 questions (in production each question is embedded when it arrives).
    VECS = [d.embedding for d in client.embeddings.create(model=EMBED, input=[r["question"] for r in REPLAY]).data]
    BUDGET["sent"] += 1
    t0 = time.perf_counter()
    client.embeddings.create(model=EMBED, input=REPLAY[0]["question"]); BUDGET["sent"] += 1
    embed_latency = time.perf_counter() - t0            # what a cache hit costs: one embedding call
    cache, seen = ResponseCache(THRESHOLD), set()
    for item, vec in zip(REPLAY, VECS):
        entry, kind, score = cache.lookup(item["question"], vec)
        row = {**item, "kind_hit": kind, "score": score, "should_hit": item["intent"] in seen}
        if entry is not None:
            row.update(hit=True, false_hit=entry["intent"] != item["intent"], latency_s=embed_latency,
                       matched=entry["question"])
        else:
            rec = await measure_stream(MOE, prefix_first(WARM_TAG, item["question"]), max_tokens=120)
            cache.add(item["question"], vec, rec["text"], item["intent"])
            row.update(hit=False, false_hit=False, latency_s=(rec["total_s"] or 0) + embed_latency, matched=None)
            await asyncio.sleep(LONG_PAUSE_S)
        seen.add(item["intent"])
        replay_rows.append(row)
df_replay = pd.DataFrame(replay_rows)
if len(df_replay):
    hits = df_replay.hit.sum()
    print(f"hit rate {hits / len(df_replay):.0%} ({hits}/50), false hits {df_replay.false_hit.sum()}, "
          f"missed chances {(df_replay.should_hit & ~df_replay.hit).sum()}, best possible hit rate {df_replay.should_hit.mean():.0%}")
    print(f"mean latency with cache {df_replay.latency_s.mean():.2f} s vs misses only {df_replay[~df_replay.hit].latency_s.mean():.2f} s")
    display(df_replay[df_replay.hit][["id", "question", "kind_hit", "score", "false_hit", "matched"]].round(3))

In [ ]:
# Threshold sweep, offline: replays the same 50 questions with the embeddings you already have (no requests).
sweep = pd.DataFrame()
if RUN_PART_B and VECS is not None:
    rows = []
    for th in np.round(np.arange(0.60, 0.99, 0.02), 2):
        c, n_hit, n_false = ResponseCache(th), 0, 0
        for item, vec in zip(REPLAY, VECS):
            entry, kind, _ = c.lookup(item["question"], vec)
            if entry is None:
                c.add(item["question"], vec, "", item["intent"])
            else:
                n_hit += 1; n_false += entry["intent"] != item["intent"]
        rows.append({"threshold": th, "hit_rate": n_hit / len(REPLAY), "false_hit_rate": n_false / len(REPLAY)})
    sweep = pd.DataFrame(rows)
    fig, ax = plt.subplots(figsize=(7, 3.6))
    ax.plot(sweep.threshold, sweep.hit_rate, "-o", color="#2a78d6", lw=2, ms=5, label="hit rate (all hits)")
    ax.plot(sweep.threshold, sweep.false_hit_rate, "-o", color="#eb6834", lw=2, ms=5, label="false-hit rate")
    ax.axvline(THRESHOLD, color="#888888", ls=":", lw=1)
    ax.set_xlabel("cosine threshold"); ax.set_ylabel("share of the 50 questions"); ax.set_ylim(0, 1)
    ax.set_title("Semantic cache: more hits means more wrong hits"); ax.grid(alpha=0.3); ax.legend(); plt.show()

In [ ]:
if RUN_PART_B:
    ckb.check("replay_complete", len(df_replay) == 50, 2, "The replay should process all 50 questions.")
    ckb.check("threshold_sweep", len(sweep) >= 10, 1, "Run the sweep cell.")
    ckb.check("false_hits_measured", len(df_replay) == 50 and "false_hit" in df_replay, 2, "Replay must track false hits.")
    if len(df_replay):
        ck.record("cache_hit_rate", float(df_replay.hit.mean()))
        ck.record("cache_false_hits", int(df_replay.false_hit.sum()))
        ck.record("cache_threshold", THRESHOLD)
bonus_got, bonus_max = ckb.score()
ck.record("bonus_score", bonus_got)
ck.record("bonus_max", 10)
ck.record("requests_sent", BUDGET["sent"])
print(f"Bonus (Part B) auto-check points: {bonus_got}/10   |   requests this notebook sent: {BUDGET['sent']}")

## Written analysis

Use **your own numbers** and cite Chapter 9 by section name or concept where asked.

**Q1. Prompt caching and prefix order.** Report median TTFT for cold, warm and question-first ("last") requests, and your speedup. Explain *why* prefix order matters, using what the KV cache stores (Chapter 9, "Prompt caching" and the KV cache discussion under attention optimization). What did the `backend` column show for warm `qwen-moe` requests, and how does `qwen-small` (one Mac) compare? What would you change in the gateway or in Larkspur's prompt layout to get more cache hits? (Think about what happens if someone puts today's date or the user's name at the top of the system prompt.)

*Your answer:*

**Q2. Batching and output length.** Using your A2 and A3 tables: how much wall time did concurrency save for 8 tickets, and what happened to per-request latency? Relate this to continuous batching (Chapter 9, "Batching"). Then compare `max_tokens` with the concise instruction on output tokens, latency, cost per 1,000 requests, and answer quality (quote one truncated ending). Which would you ship for Larkspur's technicians, and with what safety cap?

*Your answer:*

**Q3. Routing and quantization.** From your A4 tally: how much cost and latency did routing save compared with all `qwen-moe`, and what did it cost in correctness (check a few answers by hand, since the keyword check is crude)? Name one question type your heuristic would route wrongly. From A5: why does it make sense to serve `qwen-dense` at about 4 bits but `qwen-moe` at about 8 bits on a 96 GB Mac? Use the decode ceilings and, if you have them, your Lab 2 numbers. Cite Chapter 9's "Model compression" discussion of quantization.

*Your answer:*

**Q4. Recommendation for Larkspur.** Rank the optimizations you tested (prompt caching, concurrency/batching, output length control, routing, quantization; and the response cache if you did Part B) by how much they would help Larkspur's help-desk assistant. Back each rank with one of your numbers, and say what each one costs (complexity, quality risk, or a constraint it puts on prompt design). If you did Part B, include your hit rate and false-hit count and the threshold you would choose.

*Your answer:*

## AI-use reflection

Answer briefly but specifically.

1. What did you use AI assistance for in this lab (tools, which steps)?

*Your answer:*

2. Where did the AI help most, and where was it wrong or unhelpful? How did you notice?

*Your answer:*

3. What is one thing you now understand that you could explain without AI help?

*Your answer:*

4. Lab-specific: did an AI assistant suggest a prompt layout or code change that would quietly break prompt caching (for example a timestamp, request id, or the question at the top of the prompt)? How would you spot that from your measurements?

*Your answer:*

## Submit

Run this last, save, upload the .ipynb.

In [ ]:
ck.report();